In [2]:
# ============
# Common
# ============

import sys
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd()).parent
# Connect to custom-defined modules
sys.path.append(str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

In [6]:
# ========================
# Test Turnaround Decision
# ========================
from datetime import datetime
from airport_ai.decision.turnaround.structures import SafetyEvent

event = SafetyEvent(
    timestamp=datetime.now(),
    track_id=1,
    object_type="person",
    event_type="Safety Zone Violation",
    severity="HIGH",
    message="Person entered aircraft safety zone."
)

print(event)

SafetyEvent(timestamp=datetime.datetime(2026, 7, 27, 16, 48, 46, 338730), track_id=1, object_type='person', event_type='Safety Zone Violation', severity='HIGH', message='Person entered aircraft safety zone.')


In [10]:
# =========================
# Test Aircraft Selection
# =========================
from airport_ai.decision.turnaround.aircraft import AircraftSelector
from airport_ai.tracking.structures import TrackedObject

selector = AircraftSelector()

objects = [
    TrackedObject(
        track_id=1,
        class_id=0,
        class_name="person",
        confidence=0.91,
        x1=10,
        y1=10,
        x2=80,
        y2=180,
        center_x=45,
        center_y=95,
        width=70,
        height=170,
    ),
    TrackedObject(
        track_id=7,
        class_id=4,
        class_name="airplane",
        confidence=0.99,
        x1=400,
        y1=200,
        x2=1200,
        y2=650,
        center_x=800,
        center_y=425,
        width=800,
        height=450
    ),
]

aircraft = selector.select(objects)

print(aircraft)

TrackedObject(track_id=7, class_id=4, class_name='airplane', confidence=0.99, x1=400, y1=200, x2=1200, y2=650, center_x=800, center_y=425, width=800, height=450)


In [13]:
# ================================
# Test Zone Generation
# ================================
from airport_ai.decision.turnaround.zone import SafetyZoneGenerator

generator = SafetyZoneGenerator(margin_x=120, margin_y=80)

aircraft = TrackedObject(
    track_id=1,
    class_id=4,
    class_name="airplane",
    confidence=0.99,
    x1=400,
    y1=200,
    x2=1200,
    y2=700,
    center_x=800,
    center_y=450,
    width=800,
    height=500
)

zone = generator.generate(aircraft)
print(zone)


SafetyZone(x1=280, y1=120, x2=1320, y2=780, center_x=800.0, center_y=450.0, width=1040, height=660)


In [ ]:
# ===========================
# Test Visualizer
# ===========================
from airport_ai.decision.turnaround.visualization import TurnaroundVisualizer
visualizer = TurnaroundVisualizer()
frame = visualizer.draw_zone(frame, safety_zone)

In [ ]:
# ================================
# Complete Turnaround Script
# ================================
import cv2
from airport_ai.config.settings import *

from airport_ai.streams.camera import AsyncCamera

from airport_ai.tracking.tracker import YOLOTracker
from airport_ai.tracking.parser import TrackingParser

from airport_ai.decision.turnaround.aircraft import AircraftSelector
from airport_ai.decision.turnaround.zone import SafetyZoneGenerator
from airport_ai.decision.turnaround.evaluator import TurnaroundEvaluator

camera = AsyncCamera(
    source=VIDEO_SOURCE,
    width=FRAME_WIDTH,
    height=FRAME_HEIGHT,
    queue_size=FRAME_QUEUE_SIZE
).start()

tracker = YOLOTracker(str(PROJECT_ROOT/"models/yolov8n.pt"))
parser = TrackingParser()

selector = AircraftSelector()

zone_generator = SafetyZoneGenerator(
    margin_x=ZONE_MARGIN_X,
    margin_y=ZONE_MARGIN_Y
)

evaluator = TurnaroundEvaluator()

while True:
    frame = camera.read()
    result = tracker.track(frame)
    tracked_objects = parser.parse(result)
    aircraft = selector.select(tracked_objects)
    annotated = result.plot()
    if aircraft is not None:
        zone = zone.generator.generate(aircraft)
        events = evaluator.evaluate(tracked_objects, zone)
        print("="*50)
        for event in events:
            print(event)
    cv2.imshow("Turnaround Safety", annotated)
    if cv2.waitKey(1) == ord("q"):
        break
camera.stop()
cv2.destroyAllWindows()